# Discover and inspect MarketForge datasets

Run with the MarketForge research environment from this notebook's directory or the repository root. The tiny local fixtures use the normalized and physical combined layouts. They contain synthetic prices and quantities, and do not require a database or processing engines.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import runpy

from IPython.display import display
from forge.datasets import MarketData
from forge.lab.inspection import compare, inspect

fixtures = Path("dataset_fixtures.py")
if not fixtures.is_file():
    fixtures = Path("docs/content/examples/notebooks/dataset_fixtures.py")
demo = runpy.run_path(str(fixtures))
workspace = TemporaryDirectory(prefix="marketforge-notebook-")
data_root = Path(workspace.name)
demo["build"](data_root)

market = MarketData(data_root=data_root)
catalog = market.discover(scope="committed")
display(catalog)

## Search file metadata

Market predicates are matched against the same concrete membership. An OKX ETH query is empty even though the combined file contains Binance ETH and OKX BTC. The file envelope overlaps the requested half-open interval. This search reads no rows.

In [ ]:
matches = catalog.find(
    exchange=["binance", "okx"],
    instrument_type="perpetual",
    market_category="linear",
    data_type=["trades", "depth"],
    start="2026-10-27T00:00:00.000000001Z",
    end="2026-10-27T00:00:00.000000004Z",
)
display(matches)
display(matches.summary().select("operation", "data_type", "rows", "metadata_scope"))
assert not catalog.find(exchange="okx", symbol="ETHUSDT")
assert len(catalog.find(symbol="ETH", symbol_match="contains")) == 2

## Choose a file and select its stream

The combined publication has one physical file in this example. Its registered dataset ID is unknown in filesystem-only discovery. Select a stream to read its actual matching rows.

In [ ]:
combined = matches.find(operation="merge").one()
display(combined)
display(combined.streams())
trade_stream = combined.select(exchange="binance", symbol="ETHUSDT", kind="trades")
report = inspect(trade_stream)
display(report)
assert report.summary()["physical_rows"] == 3
assert report.summary()["matching_rows"] is None
display(report.schema(), report.columns()[:3], report.row_groups(), report.timestamps())

## Request exact previews, Arrow batches, and lazy queries

The preview keeps nested trade prices as exact decimal strings. Arrow returns Decimal256 values directly. The projected Polars query stays lazy until collected and applies both the stream and event-time filters.

In [ ]:
preview = trade_stream.head(2, columns=["event_timestamp_ns", "event_kind", "trade"])
display(preview)
batch = next(trade_stream.batches(columns=["trade"], batch_size=1))
assert batch.column(0)[0].as_py()["price"] == demo["PRICE"]

query = trade_stream.scan(columns=["event_timestamp_ns", "symbol"], start=demo["START"] + 1)
display(query.limit(2).collect())

## Compare metadata and profile a physical prefix

Inspection uses stored footer statistics. An explicit profile examines a physical prefix before applying any stream selector. Neither establishes continuous coverage.

In [ ]:
normalized = matches.find(operation="normalize", data_type="trade").one()
profile = inspect(normalized).profile(["price", "sequence"], input_rows=2)
display(profile)
assert profile["columns"]["price"]["min"] == demo["PRICE"]
display(compare(normalized, combined)["comparisons"])

unknown = catalog.find(path=data_root / "datasets/unknown.parquet").one()
display(unknown, unknown.head(1))
display(catalog.diagnostics)

## Open a physical part and query combined events

Locators are scoped to the current handle. The event kind and half-open time filter leave causal order intact.

In [ ]:
part = combined.partitions()[0]
display(part.info(), part.row_groups())
display(part.row_group(0).head(3, columns=["event_timestamp_ns"]))
trades = combined.query(data_type="trades", partition=part.reference)
depth = combined.query(data_type="depth", partition=part.reference)
display(trades.head(2), depth.head(2))
lazy = trades.scan(columns=["event_timestamp_ns", "stream_id"], start=demo["START"] + 1)
display(lazy.limit(2).collect(), combined.lineage())